# Clase 9 — Asistente de voz conversacional (STT + LLM + TTS)

## Objetivos de aprendizaje

- Entender el pipeline completo de un asistente de voz: voz → texto → respuesta → voz.
- Reconocer que un asistente de voz no es "un modelo", sino tres piezas independientes encadenadas.
- Ejecutar en Colab, 100% gratis, un pipeline STT (Whisper) + LLM (SmolLM2) + TTS (gTTS).
- Identificar el hueco real que quedó sin resolver en el material fuente de IBM (`text_to_speech`).

## Qué vamos a construir

Un mini asistente de voz de punta a punta: generamos un audio con una pregunta,
lo transcribimos, se lo pasamos a un LLM, y convertimos su respuesta de nuevo en
audio. Es exactamente la misma arquitectura del proyecto `chatapp-with-voice-and-openai-outline`
de IBM (`worker.py` + `server.py`), pero con piezas gratuitas de Hugging Face en
vez de Watson y OpenAI.

## Apuntes: el pipeline de un asistente de voz

Un asistente de voz conversacional encadena tres funciones independientes,
tal como se organizan en el `worker.py` real del proyecto de IBM:

1. **`speech_to_text(audio)`** — Speech-to-Text: convierte audio en texto.
   En el material original usa IBM Watson STT; aquí usamos Whisper-tiny vía
   Hugging Face.
2. **`llm_process_message(texto)`** — el LLM piensa una respuesta en texto.
   En el material original usa la API de OpenAI; aquí usamos un modelo local
   pequeño (`SmolLM2-360M-Instruct`).
3. **`text_to_speech(texto)`** — Text-to-Speech: convierte la respuesta en
   audio. En el material original usa IBM Watson TTS; aquí usamos `gTTS`
   (Google Text-to-Speech, gratis, solo necesita internet).

Un servidor Flask (`server.py`) conecta estas tres piezas con el navegador,
pero en este notebook las vamos a encadenar directamente en Python.

In [ ]:
# Instalamos las librerías que nos hacen falta (gTTS para TTS, transformers/torch
# para STT y el LLM). En Colab, transformers y torch ya suelen venir preinstalados.
!pip install -q gTTS transformers accelerate

### Paso 0: generamos un audio de "usuario" para tener algo que transcribir

En una app real, este audio vendría del micrófono del usuario grabado en el
navegador. Aquí, para poder ejecutar todo el pipeline en un notebook sin
micrófono, generamos nosotros mismos un audio de ejemplo con gTTS — así
simulamos la voz de un usuario preguntando algo en inglés.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

texto_usuario = "What is the capital of France?"
gTTS(text=texto_usuario, lang="en").save("audio_usuario.mp3")

# Reproducimos el audio generado para comprobar que suena bien
Audio("audio_usuario.mp3")

### Paso 1: Speech-to-Text (STT) — voz a texto

Usamos el mismo modelo que ya conoces del capítulo de Whisper:
`openai/whisper-tiny.en` vía `transformers.pipeline`. Es pequeño, corre en
CPU, y es la vía que usan los proyectos reales de IBM para Colab gratuito.

In [ ]:
from transformers import pipeline

stt_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
)

def speech_to_text(audio_path):
    """Recibe la ruta de un archivo de audio y devuelve el texto transcrito."""
    resultado = stt_pipe(audio_path)
    return resultado["text"].strip()

texto_transcrito = speech_to_text("audio_usuario.mp3")
print("Texto transcrito por Whisper:", texto_transcrito)

### Paso 2: el LLM — texto a texto

En el material original de IBM, esta pieza es `openai_process_message`,
que llama a la API de OpenAI (de pago). Aquí usamos un modelo pequeño y
gratuito de Hugging Face, `HuggingFaceTB/SmolLM2-360M-Instruct`, que ya
usaste en la clase de chatbots.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def llm_process_message(user_message, system_prompt="Eres un asistente breve y amable."):
    """Envía el mensaje del usuario al LLM y devuelve la respuesta en texto."""
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    entrada = tokenizer.apply_chat_template(
        mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    with torch.inference_mode():
        salida = llm_model.generate(entrada, max_new_tokens=80, do_sample=True, temperature=0.7)
    texto_generado = tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True)
    return texto_generado.strip()

respuesta = llm_process_message(texto_transcrito)
print("Respuesta del LLM:", respuesta)

### Paso 3: Text-to-Speech (TTS) — texto a voz

Esta es, precisamente, la pieza que en el `worker.py` real de IBM se quedó
con un `base_url = '...'` de relleno, sin conectar de verdad con el servicio
de pago (Watson TTS). Aquí la completamos con `gTTS`, que no necesita
ninguna clave de API.

In [ ]:
def text_to_speech(text, salida_path="respuesta.mp3", idioma="en"):
    """Convierte texto en un archivo de audio y devuelve la ruta del archivo."""
    voz = gTTS(text=text, lang=idioma)
    voz.save(salida_path)
    return salida_path

ruta_audio_respuesta = text_to_speech(respuesta)
Audio(ruta_audio_respuesta)

## Referencia: así se ve con las APIs de pago del material oficial de IBM

El código de abajo es el real de `worker.py` en el proyecto
`chatapp-with-voice-and-openai-outline`. No lo ejecutes: necesitarías
credenciales de IBM Watson y de OpenAI. Se muestra solo para que reconozcas
que es exactamente la misma arquitectura de tres funciones, hablando con
servicios externos de pago en vez de con modelos locales.

```python
# worker.py (fragmento real, requiere credenciales)

def speech_to_text(audio_binary):
    api_url = base_url + '/speech-to-text/api/v1/recognize'
    params = {'model': 'en-US_Multimedia'}
    response = requests.post(api_url, params=params, data=audio_binary).json()
    text = 'null'
    while bool(response.get('results')):
        text = response.get('results').pop().get('alternatives').pop().get('transcript')
    return text

def openai_process_message(user_message):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.choices[0].message.content

def text_to_speech(text, voice=""):
    # <-- en el material fuente, esta función solo tenía "base_url = '...'"
    #     como placeholder, sin terminar de conectar con el servicio real.
    api_url = base_url + '/text-to-speech/api/v1/synthesize?output=output_text.wav'
    if voice not in ("", "default"):
        api_url += "&voice=" + voice
    headers = {'Accept': 'audio/wav', 'Content-Type': 'application/json'}
    response = requests.post(api_url, headers=headers, json={'text': text})
    return response.content
```

**Nota importante**: en el repositorio real de IBM, `text_to_speech` quedó
sin completar (con un placeholder). Vas a resolver tú ese mismo hueco, con la
librería gratuita, en el notebook de práctica de esta clase.

## Resumen de la clase

- Un asistente de voz conversacional = STT + LLM + TTS, tres piezas
  independientes encadenadas por un servidor (o, aquí, por unas pocas líneas
  de Python).
- Cada pieza solo necesita saber el formato de entrada/salida de la anterior
  y la siguiente: audio → texto → texto → audio.
- Pudimos ejecutar el pipeline completo gratis con Whisper-tiny (STT),
  SmolLM2 (LLM) y gTTS (TTS).
- En el material real de IBM, la función `text_to_speech` quedó con un
  placeholder sin resolver — un hueco genuino del laboratorio original.

Sigue con:
- Capítulo del manual: `manual/13_asistente_voz_conversacional.md`
- Notebook de práctica: `notebooks_practica/clase09_practica_asistente_voz_conversacional.ipynb`